In [1]:
#When vertex labels fixed
def qubo_formulation_fixed (G, a, d, v_labels):
    start_time = time.process_time()
    n = G.order()
    m = G.size()

    #z2 is the number of binary bits to represent the maximum edge value possible
    z = a + (n-2)*d
    z2 = math.ceil(math.log2(z))  
    
    i_m = nx.incidence_matrix(G).toarray()
    Q = {}
    offset = 0
    #F1^2
    #for each vertex:
    for i in range(n):
        #degree
        delt_v = sum(i_m[i])
        offset += (delt_v - v_labels[i])**2
        #F1^2
        for j in range(m):       
            mij = i_m[i][j]
            if mij == 1:   
                for k in range(z2):
                    #linear part
                    if Q.get((f'y{j}{k}',f'y{j}{k}')) == None:
                        Q[(f'y{j}{k}',f'y{j}{k}')]= (2**k)**2+2*(delt_v-v_labels[i])*(2**k)
                    else:
                        Q[(f'y{j}{k}',f'y{j}{k}')]+= (2**k)**2+2*(delt_v-v_labels[i])*(2**k)

                    #quadratic part
                    for jj in range(j,m):
                        mijj=i_m[i,jj]
                        if mijj == 1:
                            for kk in range(z2):     
                                if k<kk <= z2 or j<jj<=m:
                                    if Q.get((f'y{j}{k}',f'y{jj}{kk}')) == None:
                                        Q[(f'y{j}{k}',f'y{jj}{kk}')]= 2*(2**k)*(2**kk)
                                    else:
                                        Q[(f'y{j}{k}',f'y{jj}{kk}')]+= 2*(2**k)*(2**kk)
    #in microseconds                                    
    elapsed_time = (time.process_time() - start_time)*(10**6)
    return Q,  offset, elapsed_time

In [8]:

def write_qubos_fixed(input_file,output_file):
    qubos = []
    formulation_time_list = []
    offsets = []
    new_df = pd.DataFrame()
    #new_df = pd.read_csv(output_file)
    df = pd.read_csv(input_file)
    n = len(df.index)
    randomList = random.sample(range(0, n),30)
    randomList.sort()
    #randomList = new_df['graph_num'].tolist()
    #ass = []
    #dss = []
    #ess = []
    for i in randomList:
        a = df['a'][i]
        d = df['d'][i]
        #ass.append(a)
        #dss.append(d)
        #ess.append(df['graph'][i])
        ad_str = df['Adjacency_list'][i]
        ad_list=json.loads(ad_str)
        ad = pd.DataFrame(ad_list)
        G = nx.from_pandas_adjacency(ad)
        vlabels = df['Vertex_labels'][i]
        v_labels = ast.literal_eval(vlabels)

        #do qubo formulation for 5 times to get the average time it takes
        n = 5
        times = 0
        for i in range(n):
            Q,  offset, formulation_time = qubo_formulation_fixed(G, a, d, v_labels)
            times += formulation_time
        
        qubos.append(Q)
        formulation_time_list.append(times/n)
        offsets.append(offset)
    new_df['graph_num'] = randomList
    new_df['qubo_fixed'] = qubos
    new_df['offset'] = offsets
    new_df['formulation_time/microseconds'] = formulation_time_list
    #new_df['a'] = ass
    #new_df['d'] = dss
    #new_df['e'] = ess
    #new_df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)
    new_df.to_csv(output_file)

In [10]:
import time
import math
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
import random
import ast
write_qubos_fixed('apgs/base_apgs/apg7_base.csv','qubos/fixed/apg7_fixed_qubo.csv')